# RLHF Live Coding Seminar

Этот ноутбук — сценарий семинара: заполняем TODO, прогоняем пайплайн и обсуждаем, как SFT → Reward Model → PPO с KL-штрафом выравнивает ответы.

> Советы: двигайтесь сверху вниз, не бойтесь менять гиперпараметры прямо на встрече. Все задачи маленькие, чтобы успеть в реальном времени.


## План
- Настроить окружение и пути
- Сгенерировать синтетические данные (SFT пары + preferences)
- Обучить SFT (reference policy)
- Обучить Reward Model на предпочтениях
- Запустить PPO с KL-штрафом и посмотреть, как меняются вероятности
- Поиграться с β и обсудить reward hacking

> TODO метки отмечают места для совместного дописывания.


In [ ]:
# (Опционально) Установка зависимостей для чистого окружения
# Запусти при отсутствии numpy/matplotlib
# !pip install -q numpy matplotlib


In [ ]:
# Настройка путей и импортов
from pathlib import Path
import sys

BASE = Path('.').resolve()
if not (BASE / 'simple_text_env.py').exists():
    raise RuntimeError('Запусти ноутбук из каталога code/15_rlhf_basics')

if str(BASE) not in sys.path:
    sys.path.append(str(BASE))

print('BASE:', BASE)


In [ ]:
# TODO-1: Сгенерировать синтетические данные (SFT пары + preferences)
from generate_data import main as generate_data

data_dir = BASE / 'data'
# По умолчанию создаётся 80 preference-пар (num_samples=80).
# Можно менять прямо в вызове:
# generate_data(output_dir=data_dir, num_samples=120)
# или поправить дефолт в generate_data.py
# В реальном времени сними комментарий, чтобы создать данные:
# generate_data(output_dir=data_dir, num_samples=80)

data_dir


In [ ]:
# TODO-2: Обучить SFT (reference policy)
from sft_model import train_sft, TabularPolicy

sft_ckpt = data_dir / 'sft_policy.npy'
# Раскомментируй для запуска обучения (кросс-энтропия по табличной политике)
# sft_model = train_sft(sft_path=data_dir / 'sft_data.json', ckpt_path=sft_ckpt)

# Подсмотреть logits/распределения после обучения (когда чекпоинт есть)
if sft_ckpt.exists():
    sft_model = TabularPolicy.load(sft_ckpt)
    print('SFT probs per prompt:')
    from simple_text_env import PROMPTS, CANDIDATES
    for i, p in enumerate(PROMPTS):
        probs = sft_model.predict_probs(i)
        print(f'Prompt: {p}')
        for c, pr in zip(CANDIDATES, probs):
            print(f'  {pr:0.3f} -> {c}')
        print()
else:
    print('SFT чекпоинт не найден, обучи модель выше.')


In [ ]:
# TODO-3: Обучить Reward Model на предпочтениях
from reward_model import train_reward_model, RewardModel

rm_ckpt = data_dir / 'reward_model.npy'
# Раскомментируй, чтобы обучить RM на preferences.json
# rm = train_reward_model(pref_path=data_dir / 'preferences.json', ckpt_path=rm_ckpt)

if rm_ckpt.exists():
    rm = RewardModel.load(rm_ckpt)
    print('Reward weights:', rm.w)
else:
    print('RM чекпоинт не найден, обучи модель выше.')


In [ ]:
# TODO-4: Запустить PPO c KL-штрафом
from ppo_rlhf import run as run_ppo

beta = 0.01  # Поиграйтесь: 0.0, 0.001, 0.01, 0.1
# Раскомментируй строку ниже для старта RL fine-tuning
# run_ppo(beta=beta, output_dir=data_dir)

print('Готово к запуску PPO; выбери beta и сними комментарий выше.')


In [ ]:
# TODO-5: Посмотреть, как изменилась политика после PPO
from simple_text_env import PROMPTS, CANDIDATES
import numpy as np

ppo_ckpt = data_dir / f'ppo_actor_beta_{beta}.npy'
if ppo_ckpt.exists():
    from sft_model import TabularPolicy
    actor = TabularPolicy()
    actor.logits = np.load(ppo_ckpt)
    print(f'Проверяем ppo_actor_beta_{beta}.npy')
    for i, prompt in enumerate(PROMPTS):
        probs = actor.predict_probs(i)
        print(f'Prompt: {prompt}')
        for resp, p in zip(CANDIDATES, probs):
            print(f'  {p:0.3f} -> {resp}')
        print()
else:
    print('Запусти PPO выше, чтобы появился чекпоинт.')


In [ ]:
# Визуализация: сравнение распределений SFT vs PPO
import numpy as np
import matplotlib.pyplot as plt
from sft_model import TabularPolicy
from simple_text_env import PROMPTS, CANDIDATES

sft_ckpt = data_dir / 'sft_policy.npy'
ppo_ckpt = data_dir / f'ppo_actor_beta_{beta}.npy'

if not sft_ckpt.exists() or not ppo_ckpt.exists():
    print('Нужны чекпоинты sft_policy.npy и ppo_actor_beta_{beta}.npy. Запусти TODO-2 и TODO-4.')
else:
    sft = TabularPolicy.load(sft_ckpt)
    ppo = TabularPolicy()
    ppo.logits = np.load(ppo_ckpt)

    fig, axes = plt.subplots(len(PROMPTS), 1, figsize=(8, 10), sharex=True)
    axes = np.atleast_1d(axes)
    x = np.arange(len(CANDIDATES))

    for i, ax in enumerate(axes):
        sft_probs = sft.predict_probs(i)
        ppo_probs = ppo.predict_probs(i)
        ax.bar(x - 0.15, sft_probs, width=0.3, label='SFT')
        ax.bar(x + 0.15, ppo_probs, width=0.3, label='PPO')
        ax.set_title(PROMPTS[i])
        ax.set_ylim(0, 1)
        ax.set_xticks(x)
        ax.set_xticklabels([f'a{j}' for j in range(len(CANDIDATES))])
        ax.legend()

    plt.tight_layout()
    plt.show()

